**Actividad grupal: Sistemas de recomendación –
Parte I**

El objetivo de la actividad es evaluar la comprensión del alumno de los temas presentados en los apuntes
mediante la implementación de funciones Python para el desarrollo de un algoritmo de recomendación. El
objetivo es que los estudiantes apliquen los conceptos aprendidos sobre tipos de sistemas de
recomendación, evaluación de los mismos en la implementación práctica de un sistema de filtrado
colaborativo.

**Consigna**

Cada estudiante deberá desarrollar un sistema de recomendación basado en filtrado colaborativo
utilizando Python y una biblioteca de su elección (Scikit-Learn o Surprise) que cumpla con las siguientes
consignas:
1. **Preparación de los datos:** Vamos a utilizar el dataset MovieLens (https://grouplens.org/datasets/movielens/). Para ello,
podemos descargarlo del sitio, seleccionando un conjunto de datos de los disponibles para
descargar. Por ejemplo, "ml-latest-small.zip" es una versión reducida del conjunto de datos.
(es posible cargar el dataset en memoria, utilizando pandas y el método read_csv()).
2. **Implementación del filtrado colaborativo basado en usuario o ítem:** Elegir entre el filtrado colaborativo basado en usuario o en ítem e implementar el algoritmo
correspondiente utilizando la biblioteca seleccionada.
3. **Evaluación del sistema de recomendación:** Utilizando las métricas RMSE, MAE y precisión, evaluar el modelo creado.
4. **Probar el sistema de recomendación:** Generar recomendaciones utilizando el modelo creado para un usuario objetivo. Es importante que los alumnos describan con celdas de texto lo que van desarrollando y
además que realicen un análisis de los resultados obtenidos.

**Preparación de los datos**

Primero descargamos el archivo ml-latest-small.zip con los datos de MovieLens. Luego en la seccion de files creamos una carpeta ml-latest-small donde copiamos los archivos csv a usar.

A continuacion cargamos los datos usando Pandas.

In [ ]:
import pandas as pd

links = pd.read_csv("ml-latest-small/ratings.csv")
movies = pd.read_csv("ml-latest-small/movies.csv")
ratings = pd.read_csv("ml-latest-small/ratings.csv")
tags = pd.read_csv("ml-latest-small/tags.csv")

Verificamos que los datos hayan sido cargados correctamente

In [ ]:
display(links.head())
display(movies.head())
display(ratings.head())
display(tags.head())

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


**Implementación del filtrado colaborativo basado en usuario o ítem**

Implementamos un sistema de filtrado colaborativo basado en items

In [ ]:
!pip install scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp310-cp310-linux_x86_64.whl size=2357269 sha256=dde119a6a612db55ed3a9f6869751831cd994ecd9b8ad4a7d2566dbc19bbcc14
  Stored in directory: /root/.cache/pip/wheels/4b/3f/df/6acbf0a40397d9bf3ff97f582cc22fb9ce66adde75bc71fd54
Successfully built scikit-surprise


In [ ]:
from surprise import Reader, Dataset, KNNBasic
from surprise.model_selection import train_test_split

# El raging scale define el rango de valores que pueden tomar las calificaciones
# En el caso de MovieLens el rango de valores va de 0.5 a 5
reader = Reader(rating_scale=(0.5, 5))

# Carga los datos
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# Dividir los datos en conjuntos de entrenamiento y prueba
trainset, testset = train_test_split(data, test_size=0.25)

# Usar el algoritmo KNN basado en items para el filtrado colaborativo
algo = KNNBasic(sim_options={'user_based': False})

# Entrenar el modelo usando los datos de entrenamiento
algo.fit(trainset)

# Realizar predicciones usando los datos prueba
predictions = algo.test(testset)

Computing the msd similarity matrix...
Done computing similarity matrix.


**Evaluación del sistema de recomendación**

Utilizamos las métricas RMSE, MAE y precisión para evaluar el modelo creado.

In [ ]:
from surprise import accuracy

# Calcular el RMSE (Root Mean Squared Error)
rmse = accuracy.rmse(predictions)

# Calcular el MAE (Mean Absolute Error)
mae = accuracy.mae(predictions)

RMSE: 0.9078
MAE:  0.7006


In [ ]:
from collections import defaultdict

# Definimos la siguiente funcion para calcular la precision: https://surprise.readthedocs.io/en/stable/FAQ.html#precision-recall-at-k-py
def precision_recall_at_k(predictions, k=10, threshold=3.5):
    """Return precision and recall at k metrics for each user"""

    # First map the predictions to each user.
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()
    for uid, user_ratings in user_est_true.items():

        # Sort user ratings by estimated value
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        # Number of relevant items
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)

        # Number of recommended items in top k
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])

        # Number of relevant and recommended items in top k
        n_rel_and_rec_k = sum(
            ((true_r >= threshold) and (est >= threshold))
            for (est, true_r) in user_ratings[:k]
        )

        # Precision@K: Proportion of recommended items that are relevant
        # When n_rec_k is 0, Precision is undefined. We here set it to 0.

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0

        # Recall@K: Proportion of relevant items that are recommended
        # When n_rel is 0, Recall is undefined. We here set it to 0.

        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    return precisions, recalls

# Calculamos precision y recall con k = 10 y threshold = 3
precisions, recalls = precision_recall_at_k(predictions, k=10, threshold=3)
print(f"Precision: {sum(prec for prec in precisions.values()) / len(precisions)}")
print(f"Recall: {sum(rec for rec in recalls.values()) / len(recalls)}")


Precision: 0.8795986208691121
Recall: 0.5477523028697512


**Probar el sistema de recomendación**

Generamos recomendaciones utilizando el modelo creado para un usuario objetivo.

In [ ]:
# ID del usuario objetivo
user_id = 1

# Obtener películas que el usuario no califico
movies_not_rated = ratings[ratings['userId'] != user_id]['movieId'].unique()

# Predecir las calificaciones para las películas
predictions = [algo.predict(user_id, movie_id) for movie_id in movies_not_rated]

# Obtener las mejores predicciones
top_n_recommendations = sorted(predictions, key=lambda x: x.est, reverse=True)[:10]

# Imprimir las recomendaciones
print(f"Recomendaciones para el usuario {user_id}:")
for prediction in top_n_recommendations:
    # Buscar el título de la película
    movie_title = movies[movies['movieId'] == prediction.iid]['title'].iloc[0]
    print(f"Película: {movie_title}, Calificación: {prediction.est}")

Recomendaciones para el usuario 1:
Película: The Jinx: The Life and Deaths of Robert Durst (2015), Calificación: 5
Película: Private Lives of Pippa Lee, The (2009), Calificación: 5
Película: St Trinian's 2: The Legend of Fritton's Gold (2009), Calificación: 5
Película: Rust and Bone (De rouille et d'os) (2012), Calificación: 5
Película: Enough Said (2013), Calificación: 5
Película: Pretty One, The (2013), Calificación: 5
Película: The Second Best Exotic Marigold Hotel (2015), Calificación: 5
Película: Magic Mike XXL (2015), Calificación: 5
Película: Black Cauldron, The (1985), Calificación: 5
Película: Great Mouse Detective, The (1986), Calificación: 5
